# NegLogProbLoss Scaling Checks

This notebook verifies how `NegLogProbLoss` scales likelihood terms for train/validation/test splits and mini-batches. Each code cell compares the loss value against a direct manual calculation from the relevant pointwise log-probability array and asserts that the values match.

In [1]:
from types import SimpleNamespace

import jax.numpy as jnp
import pandas as pd
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl
import liesel.optim as opt
from liesel.optim.types import Position

In [2]:
def pointwise_sum(state, node_name):
    return state[node_name].value.sum()


def empty_carry(model, **kwargs):
    values = {
        "model_state": model.state,
        "fixed_position": Position({}),
        "batch": Position({}),
        "batches": None,
    }
    values.update(kwargs)
    return SimpleNamespace(**values)


def report(rows):
    out = []
    for row in rows:
        actual = float(row.pop("actual"))
        expected = float(row.pop("expected"))
        out.append(
            {
                **row,
                "actual": actual,
                "expected": expected,
                "abs_error": abs(actual - expected),
                "allclose": bool(jnp.allclose(actual, expected)),
            }
        )

    df = pd.DataFrame(out)
    assert bool(df["allclose"].all())
    return df


def normal_response(name, value, loc=0.0):
    return lsl.Var.new_obs(
        jnp.asarray(value),
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name=name,
    )

In [3]:
def simple_model(n=12):
    y = normal_response("y", jnp.linspace(-1.5, 1.5, n))
    return lsl.Model([y])


def split_array_model(n_rows=4, n_cols=10):
    X_value = jnp.linspace(-1.0, 1.0, 2 * n_cols).reshape(2, n_cols)
    y_value = jnp.linspace(-0.5, 1.5, n_rows * n_cols).reshape(n_rows, n_cols)
    X = lsl.Var.new_obs(X_value, name="X")
    loc = lsl.Var.new_calc(lambda x: x.mean(axis=0), X, name="loc")
    y = normal_response("y", y_value, loc=loc)
    return lsl.Model([y])


def batch_array_model(n_rows=3, n_cols=12):
    X_value = jnp.linspace(-2.0, 2.0, 2 * n_cols).reshape(2, n_cols)
    y_value = jnp.linspace(-1.0, 1.0, n_rows * n_cols).reshape(n_rows, n_cols)
    X = lsl.Var.new_obs(X_value, name="X")
    loc = lsl.Var.new_calc(lambda x: x.mean(axis=0), X, name="loc")
    y = normal_response("y", y_value, loc=loc)
    return lsl.Model([y])


def split_batch_array_model(n_rows=6, n_cols=8):
    X_split_value = jnp.linspace(-1.0, 1.0, n_rows * 2).reshape(n_rows, 2)
    Z_batch_value = jnp.linspace(-0.5, 0.5, 3 * n_cols).reshape(3, n_cols)
    y_value = jnp.linspace(-2.0, 2.0, n_rows * n_cols).reshape(n_rows, n_cols)
    X_split = lsl.Var.new_obs(X_split_value, name="X_split")
    Z_batch = lsl.Var.new_obs(Z_batch_value, name="Z_batch")
    loc = lsl.Var.new_calc(
        lambda x, z: x[:, :1] + z.mean(axis=0)[None, :],
        X_split,
        Z_batch,
        name="loc",
    )
    y = normal_response("y", y_value, loc=loc)
    return lsl.Model([y])

## 1. Train / Validation / Test Split, No Batching

### 1.a) Simple model

The response has one observation axis. The validation likelihood is scaled by `train_sample_size / validate_sample_size`.

In [4]:
model = simple_model(10)
split = opt.PositionSplit.from_model(
    model,
    ["y"],
    validate_axis_share=0.2,
    test_axis_share=0.1,
    shuffle=False,
)
loss = opt.NegLogProbLoss(model, split, scale=False)
scaled_loss = opt.NegLogProbLoss(model, split, scale=True)
carry = empty_carry(model)

train_state = model.update_state(split.train, model.state)
validate_state = model.update_state(split.validate, model.state)

report(
    [
        {
            "quantity": "train loss",
            "actual": loss.loss_train(Position({}), carry),
            "expected": -pointwise_sum(train_state, "y_log_prob"),
            "sample_scale": split.sample_scale("train"),
            "normalizer": loss.scalar,
        },
        {
            "quantity": "validation loss",
            "actual": loss.loss_validate(Position({}), carry),
            "expected": -split.validate_sample_scale
            * pointwise_sum(validate_state, "y_log_prob"),
            "sample_scale": split.sample_scale("validate"),
            "normalizer": loss.scalar,
        },
        {
            "quantity": "scaled train loss",
            "actual": scaled_loss.loss_train(Position({}), carry),
            "expected": -pointwise_sum(train_state, "y_log_prob")
            / split.train_sample_size,
            "sample_scale": split.sample_scale("train"),
            "normalizer": scaled_loss.scalar,
        },
    ]
)

,quantity,sample_scale,normalizer,actual,expected,abs_error,allclose
0,train loss,1.0,1.0,8.863126,8.863126,0.0,True
1,validation loss,3.5,1.0,10.029793,10.029793,0.0,True
2,scaled train loss,1.0,7.0,1.266161,1.266161,0.0,True


### 1.b) Array model

The response has shape `(n_rows, n_cols)`, but splitting happens over columns. `X` has no distribution and only shares the split axis with the response, so the inferred sample size comes from the pointwise response log-probability array.

In [5]:
model = split_array_model()
split = opt.PositionSplit.from_model(
    model,
    ["X", "y"],
    validate_axis_share=0.2,
    test_axis_share=0.1,
    split_axes={"X": 1, "y": 1},
    shuffle=False,
)
loss = opt.NegLogProbLoss(model, split, scale=False)
scaled_loss = opt.NegLogProbLoss(model, split, scale=True)
carry = empty_carry(model)

train_state = model.update_state(split.train, model.state)
validate_state = model.update_state(split.validate, model.state)

{
    "train_shapes": {key: value.shape for key, value in split.train.items()},
    "sample_sizes": split.sample_sizes,
    "checks": report(
        [
            {
                "quantity": "train loss",
                "actual": loss.loss_train(Position({}), carry),
                "expected": -pointwise_sum(train_state, "y_log_prob"),
                "sample_scale": split.sample_scale("train"),
                "normalizer": loss.scalar,
            },
            {
                "quantity": "validation loss",
                "actual": loss.loss_validate(Position({}), carry),
                "expected": -split.validate_sample_scale
                * pointwise_sum(validate_state, "y_log_prob"),
                "sample_scale": split.sample_scale("validate"),
                "normalizer": loss.scalar,
            },
            {
                "quantity": "scaled validation loss",
                "actual": scaled_loss.loss_validate(Position({}), carry),
                "expected": -split.validate_sample_scale
                * pointwise_sum(validate_state, "y_log_prob")
                / split.train_sample_size,
                "sample_scale": split.sample_scale("validate"),
                "normalizer": scaled_loss.scalar,
            },
        ]
    ),
}

{'train_shapes': {'X': (2, 7), 'y': (4, 7)},
 'sample_sizes': {'train': 28.0, 'validate': 8.0, 'test': 4.0},
 'checks':                  quantity  sample_scale  normalizer     actual   expected  \
 0              train loss           1.0         1.0  35.221088  35.221088   
 1         validation loss           3.5         1.0  31.942667  31.942667   
 2  scaled validation loss           3.5        28.0   1.140810   1.140810   
 
    abs_error  allclose  
 0        0.0      True  
 1        0.0      True  
 2        0.0      True  }

## 2. Batching, No Splitting

### 2.a) Simple model

There is no validation split. The batched training loss scales the current mini-batch likelihood by `sample_size / batch_sample_size`.

In [6]:
model = simple_model(12)
split = opt.PositionSplit.from_model(model, ["y"], shuffle=False)
batches = opt.Batches.from_model(
    model,
    batch_axis_size=4,
    position_keys=["y"],
    shuffle=False,
)
batch = batches.get_batched_position(model.extract_position(["y"]), 0)
loss = opt.NegLogProbLoss(model, split, scale=False)
scaled_loss = opt.NegLogProbLoss(model, split, scale=True)
carry = empty_carry(model, batch=batch, batches=batches)

batch_state = model.update_state(batch, model.state)

{
    "batch_sample_scale": batches.batch_sample_scale,
    "sample_size": batches.sample_size,
    "batch_sample_size": batches.batch_sample_size,
    "checks": report(
        [
            {
                "quantity": "batch loss",
                "actual": loss.loss_train_batched(Position({}), carry),
                "expected": -batches.batch_sample_scale
                * pointwise_sum(batch_state, "y_log_prob"),
                "sample_scale": batches.batch_sample_scale,
                "normalizer": loss.scalar,
            },
            {
                "quantity": "scaled batch loss",
                "actual": scaled_loss.loss_train_batched(Position({}), carry),
                "expected": -batches.batch_sample_scale
                * pointwise_sum(batch_state, "y_log_prob")
                / split.train_sample_size,
                "sample_scale": batches.batch_sample_scale,
                "normalizer": scaled_loss.scalar,
            },
        ]
    ),
}

{'batch_sample_scale': 3.0,
 'sample_size': 12.0,
 'batch_sample_size': 4.0,
 'checks':             quantity  sample_scale  normalizer     actual   expected  \
 0         batch loss           3.0         1.0  18.725609  18.725609   
 1  scaled batch loss           3.0        12.0   1.560467   1.560467   
 
    abs_error  allclose  
 0        0.0      True  
 1        0.0      True  }

### 2.b) Array model

Batching happens over columns. `X` has no distribution and only shares the batching axis with `y`. The full and batch sample sizes are inferred from the response log-probability shape.

In [ ]:
model = batch_array_model()
split = opt.PositionSplit.from_model(
    model,
    ["X", "y"],
    split_axes={"X": 1, "y": 1},
    shuffle=False,
)
batches = opt.Batches.from_model(
    model,
    batch_axis_size=4,
    position_keys=["X", "y"],
    batch_axes={"X": 1, "y": 1},
    shuffle=False,
)
batch = batches.get_batched_position(model.extract_position(["X", "y"]), 0)
loss = opt.NegLogProbLoss(model, split, scale=False)
scaled_loss = opt.NegLogProbLoss(model, split, scale=True)
carry = empty_carry(model, batch=batch, batches=batches)

batch_state = model.update_state(batch, model.state)

{
    "batch_shapes": {key: value.shape for key, value in batch.items()},
    "split_sample_sizes": split.sample_sizes,
    "batch_sample_size": batches.batch_sample_size,
    "batch_sample_scale": batches.batch_sample_scale,
    "checks": report(
        [
            {
                "quantity": "batch loss",
                "actual": loss.loss_train_batched(Position({}), carry),
                "expected": -batches.batch_sample_scale
                * pointwise_sum(batch_state, "y_log_prob"),
                "sample_scale": batches.batch_sample_scale,
                "normalizer": loss.scalar,
            },
            {
                "quantity": "scaled batch loss",
                "actual": scaled_loss.loss_train_batched(Position({}), carry),
                "expected": -batches.batch_sample_scale
                * pointwise_sum(batch_state, "y_log_prob")
                / split.train_sample_size,
                "sample_scale": batches.batch_sample_scale,
                "normalizer": scaled_loss.scalar,
            },
        ]
    ),
}

{'batch_shapes': {'X': (2, 4), 'y': (3, 4)},
 'split_sample_sizes': {'train': 36.0},
 'batch_sample_size': 12.0,
 'batch_sample_scale': 3.0,
 'checks':             quantity  sample_scale  normalizer     actual   expected  \
 0         batch loss           3.0         1.0  42.957993  42.957993   
 1  scaled batch loss           3.0        36.0   1.193278   1.193278   
 
    abs_error  allclose  
 0        0.0      True  
 1        0.0      True  }

## 3. Batching And Splitting

### 3.a) Simple model

The split defines the training sample size. The manually constructed full-training batches use that sample size for mini-batch scaling.

In [8]:
model = simple_model(12)
split = opt.PositionSplit.from_model(
    model,
    ["y"],
    validate_axis_share=0.25,
    shuffle=False,
)
batches = opt.Batches(
    ["y"],
    axis_size=split.train_axis_size,
    batch_axis_size=3,
    shuffle=False,
    sample_size=split.train_sample_size,
)
batch = batches.get_batched_position(split.train, 0)
loss = opt.NegLogProbLoss(model, split, scale=False)
scaled_loss = opt.NegLogProbLoss(model, split, scale=True)
carry = empty_carry(model, batch=batch, batches=batches)

batch_state = model.update_state(batch, model.state)
validate_state = model.update_state(split.validate, model.state)

{
    "split_sample_sizes": split.sample_sizes,
    "batch_sample_size": batches.batch_sample_size,
    "batch_sample_scale": batches.batch_sample_scale,
    "checks": report(
        [
            {
                "quantity": "batch loss",
                "actual": loss.loss_train_batched(Position({}), carry),
                "expected": -batches.batch_sample_scale
                * pointwise_sum(batch_state, "y_log_prob"),
                "sample_scale": batches.batch_sample_scale,
                "normalizer": loss.scalar,
            },
            {
                "quantity": "validation loss",
                "actual": loss.loss_validate(Position({}), empty_carry(model)),
                "expected": -split.validate_sample_scale
                * pointwise_sum(validate_state, "y_log_prob"),
                "sample_scale": split.sample_scale("validate"),
                "normalizer": loss.scalar,
            },
            {
                "quantity": "scaled batch loss",
                "actual": scaled_loss.loss_train_batched(Position({}), carry),
                "expected": -batches.batch_sample_scale
                * pointwise_sum(batch_state, "y_log_prob")
                / split.train_sample_size,
                "sample_scale": batches.batch_sample_scale,
                "normalizer": scaled_loss.scalar,
            },
        ]
    ),
}

{'split_sample_sizes': {'train': 9.0, 'validate': 3.0},
 'batch_sample_size': 3.0,
 'batch_sample_scale': 3.0,
 'checks':             quantity  sample_scale  normalizer     actual   expected  \
 0         batch loss           3.0         1.0  15.271480  15.271480   
 1    validation loss           3.0         1.0  15.271481  15.271481   
 2  scaled batch loss           3.0         9.0   1.696831   1.696831   
 
    abs_error  allclose  
 0        0.0      True  
 1        0.0      True  
 2        0.0      True  }

### 3.b) Array model with different split and batch axes

The response has shape `(n_rows, n_cols)`. Splitting acts over rows through `y` and `X_split`; batching acts over columns through `y` and `Z_batch`. `Z_batch` has no distribution and is not split, but it is batched because it shares only the batching axis with the response.

In [ ]:
model = split_batch_array_model()
split = opt.PositionSplit.from_model(
    model,
    ["X_split", "y"],
    validate_axis_share=1 / 3,
    shuffle=False,
)
Z_batch_full = model.extract_position(["Z_batch"])["Z_batch"]
batches = opt.Batches(
    ["Z_batch", "y"],
    axis_size=8,
    batch_axis_size=2,
    batch_axes={"Z_batch": 1, "y": 1},
    shuffle=False,
    sample_size=split.train_sample_size,
)
train_position_for_batching = Position(
    {
        "X_split": split.train["X_split"],
        "Z_batch": Z_batch_full,
        "y": split.train["y"],
    }
)
batch = batches.get_batched_position(train_position_for_batching, 0)
fixed_train = Position({"X_split": split.train["X_split"]})
loss = opt.NegLogProbLoss(model, split, scale=False)
scaled_loss = opt.NegLogProbLoss(model, split, scale=True)
carry = empty_carry(model, batch=batch, batches=batches, fixed_position=fixed_train)

batch_state = model.update_state(Position(batch | fixed_train), model.state)
validate_state = model.update_state(
    Position(split.validate | {"Z_batch": Z_batch_full}),
    model.state,
)

{
    "split_train_shapes": {key: value.shape for key, value in split.train.items()},
    "batch_shapes": {key: value.shape for key, value in batch.items()},
    "split_sample_sizes": split.sample_sizes,
    "batch_sample_size": batches.batch_sample_size,
    "batch_sample_scale": batches.batch_sample_scale,
    "checks": report(
        [
            {
                "quantity": "batch loss",
                "actual": loss.loss_train_batched(Position({}), carry),
                "expected": -batches.batch_sample_scale
                * pointwise_sum(batch_state, "y_log_prob"),
                "sample_scale": batches.batch_sample_scale,
                "normalizer": loss.scalar,
            },
            {
                "quantity": "validation loss",
                "actual": loss.loss_validate(
                    Position({}),
                    empty_carry(
                        model,
                        fixed_position=Position({"Z_batch": Z_batch_full}),
                    ),
                ),
                "expected": -split.validate_sample_scale
                * pointwise_sum(validate_state, "y_log_prob"),
                "sample_scale": split.sample_scale("validate"),
                "normalizer": loss.scalar,
            },
            {
                "quantity": "scaled batch loss",
                "actual": scaled_loss.loss_train_batched(Position({}), carry),
                "expected": -batches.batch_sample_scale
                * pointwise_sum(batch_state, "y_log_prob")
                / split.train_sample_size,
                "sample_scale": batches.batch_sample_scale,
                "normalizer": scaled_loss.scalar,
            },
        ]
    ),
}

{'split_train_shapes': {'X_split': (4, 2), 'y': (4, 8)},
 'batch_shapes': {'Z_batch': (3, 2), 'y': (4, 2)},
 'split_sample_sizes': {'train': 32.0, 'validate': 16.0},
 'batch_sample_size': 8.0,
 'batch_sample_scale': 4.0,
 'checks':             quantity  sample_scale  normalizer     actual   expected  \
 0         batch loss           4.0         1.0  33.398819  33.398819   
 1    validation loss           2.0         1.0  38.371948  38.371948   
 2  scaled batch loss           4.0        32.0   1.043713   1.043713   
 
    abs_error  allclose  
 0        0.0      True  
 1        0.0      True  
 2        0.0      True  }

## Summary

All code cells above contain assertions through `report(...)`. A successful notebook run means the displayed `actual` and `expected` values agree for every scaling check.

In [10]:
"All NegLogProbLoss scaling checks passed."

'All NegLogProbLoss scaling checks passed.'